# 5.2 Clasificación Ideológica de Noticias en Español
## Zero-Shot y Few-Shot
Este notebook es el **noveno paso del pipeline**, que tiene como objetivo analizar el ground truth y hacer el preprocesado para generar los modelos del clasificación ideológica

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_recall_fscore_support,
    cohen_kappa_score, confusion_matrix,
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    pipeline,
)
import subprocess, sys
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# ── Variables de entorno (definidas en 5.0_ground_truth.ipynb) ─────────────
ROOT       = Path('..')
SPLITS_DIR = ROOT / 'data' / 'splits'

STOPWORDS_ES = [
    'de','la','el','en','y','a','los','del','se','las','por','un','para','con',
    'no','una','su','al','lo','como','más','pero','sus','le','ya','o','este',
    'sí','porque','esta','entre','cuando','muy','sin','sobre','ser','tiene',
    'también','me','hasta','hay','donde','han','quien','están','estado','antes',
    'qué','esto','fue','ha','si','te','que','es','son','era','sido','ser','estar',
    'algo','todo','cada','tan','bien','vez','así','aunque','sino','tras','ante',
    'bajo','caso','año','años','según','mismo','misma','parte','puede','solo',
]

print(f'ROOT      : {ROOT.resolve()}')
print(f'SPLITS_DIR: {SPLITS_DIR}')

## 1. Zero-Shot

Evalúa 4 modelos en clasificación ideológica de medios españoles **sin fine-tuning**:

| Modelo | Tipo |
|---|---|
| `Qwen/Qwen3-0.6B` | Generativo (causal LM) |
| `Qwen/Qwen3-1.7B` | Generativo (causal LM) |
| `Qwen/Qwen3-4B` | Generativo (causal LM) |
| `PlanTL/roberta-base-bne` | Zero-shot classification pipeline |

**Métrica principal**: F1-macro. En caso de empate se prioriza Qwen3 por coherencia con el pipeline de embeddings.

### 1.1 Setup

In [6]:
OUT_ZS_DIR = ROOT / "output"
OUT_ZS_DIR.mkdir(parents=True, exist_ok=True)

# Device: Apple Silicon M3
ZS_DEVICE = "mps"
ZS_DTYPE  = torch.float16

CLASES       = ["IZQUIERDA", "CENTRO", "DERECHA"]
LABEL_MAP_ZS = {"izquierda": "IZQUIERDA", "centro": "CENTRO", "derecha": "DERECHA"}

PROMPT_TEMPLATE = (
    "/no_think\n"
    "Clasifica la ideología política de este artículo español.\n"
    "Responde con exactamente una palabra: IZQUIERDA, CENTRO o DERECHA.\n\n"
    "Artículo: {texto}\n\n"
    "Respuesta (una palabra):"
)

print(f"MPS disponible: {torch.backends.mps.is_available()}")
print(f"Output dir    : {OUT_ZS_DIR}")

MPS disponible: True
Output dir    : ../output


### 5.2 Muestra balanceada (150 artículos, 50 por clase)

In [7]:
df_val_zs = pd.read_parquet(SPLITS_DIR / "df_val.parquet")

muestras_zs = [
    df_val_zs[df_val_zs["final_label"] == cls].sample(50, random_state=42)
    for cls in CLASES
]
df_muestra = pd.concat(muestras_zs).reset_index(drop=True)

print(f"Muestra balanceada: {len(df_muestra)} artículos")
print(df_muestra["final_label"].value_counts().to_string())
print(f"\nLongitud media del texto: {df_muestra['texto'].str.len().mean():.0f} caracteres")

Muestra balanceada: 150 artículos
final_label
IZQUIERDA    50
CENTRO       50
DERECHA      50

Longitud media del texto: 3946 caracteres


 Se usa una muestra  balanceada: **50 artículos por clase** (150 en total), con una longitud media de ~3 946 caracteres. El balanceo elimina el sesgo por soporte: así el F1-macro refleja directamente la capacidad discriminativa del modelo en cada clase ideológica, sin que una clase mayoritaria infle artificialmente las métricas globales.

### 5.3 Función de métricas

In [8]:
def calcular_metricas_zs(nombre, y_true, y_pred, probs_arr):
    acc      = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro",    labels=CLASES, zero_division=0)
    f1_w     = f1_score(y_true, y_pred, average="weighted", labels=CLASES, zero_division=0)
    kappa    = cohen_kappa_score(y_true, y_pred)
    prec, rec, f1_per, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASES, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=CLASES)
    conf_global = probs_arr.max(axis=1).mean()

    print(f"\n  Accuracy    : {acc:.4f}")
    print(f"  F1-macro    : {f1_macro:.4f}")
    print(f"  F1-weighted : {f1_w:.4f}")
    print(f"  Kappa       : {kappa:.4f}")
    print(f"  Confianza   : {conf_global:.4f}")
    for i, cls in enumerate(CLASES):
        print(f"  {cls}: P={prec[i]:.3f}  R={rec[i]:.3f}  F1={f1_per[i]:.3f}")
    print("\n  Matriz de confusión (filas=real, cols=pred):")
    print(pd.DataFrame(cm, index=CLASES, columns=CLASES).to_string())

    row = {
        "modelo": nombre, "n_eval": len(y_true),
        "accuracy": round(acc, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_weighted": round(f1_w, 4),
        "kappa": round(kappa, 4),
        "confianza_media": round(conf_global, 4),
    }
    for i, cls in enumerate(CLASES):
        tag = cls.lower()[:3]
        row[f"precision_{tag}"] = round(prec[i], 4)
        row[f"recall_{tag}"]    = round(rec[i], 4)
        row[f"f1_{tag}"]        = round(f1_per[i], 4)

    df_tmp = pd.DataFrame({"pred": y_pred, "conf": probs_arr.max(axis=1)})
    for cls in CLASES:
        vals = df_tmp[df_tmp["pred"] == cls]["conf"]
        row[f"conf_{cls.lower()[:3]}"] = round(vals.mean(), 4) if len(vals) > 0 else float("nan")

    return row

### 5.4 Evaluación Qwen3 (generativo, zero-shot)

Prompt completo al tokenizer con `max_length=1024, truncation=True`.

**Decisión de diseño:**
La versión inicial usaba búsqueda de palabras clave en el texto generado (`izquierda / centro / derecha`). Esto producía dos problemas:
1. **Muchos fallbacks**: el modelo con `/no_think` genera respuestas cortas que a veces no contienen ninguna de las tres palabras exactas.
2. **CENTRO nunca aparecía**: la búsqueda recorre el dict en orden `izquierda → centro → derecha`; si el modelo razona *"no es de izquierda ni de derecha, es de centro"* matchea `izquierda` primero.

**Solución**: uso `argmax` sobre los logits del primer token generado, restringidos a los token-IDs de `izquierda`, `centro` y `derecha`. Estos logits son la señal más directa del modelo antes de cualquier decodificación, si el modelo considera CENTRO más probable, su logit ganará aunque el texto generado sea ruidoso. Esto elimina los fallbacks y permite que CENTRO compita en igualdad de condiciones.

In [24]:
def evaluar_qwen3(nombre_modelo, df_muestra):
    nombre_corto = nombre_modelo.split("/")[-1]
    print(f"{'='*64}")
    print(f"  Modelo: {nombre_modelo}")
    print(f"{'='*64}")

    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    model = AutoModelForCausalLM.from_pretrained(
        nombre_modelo,
        torch_dtype=ZS_DTYPE,
    ).to(ZS_DEVICE)
    model.eval()

    def first_token_id(word):
        return tokenizer.encode(word, add_special_tokens=False)[0]

    tok_ids = [first_token_id(w) for w in ["izquierda", "centro", "derecha"]]

    ids_out, y_true_out, preds_out = [], [], []
    probs_list = []

    for _, row in df_muestra.iterrows():
        prompt = PROMPT_TEMPLATE.format(texto=row["texto"])
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            max_length=1024,
            truncation=True,
        ).to(ZS_DEVICE)

        with torch.no_grad():
            gen_out = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=None,
                top_p=None,
                output_scores=True,
                return_dict_in_generate=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Probabilidades sobre las 3 clases (logits del 1er token generado)
        class_logits = gen_out.scores[0][0][tok_ids].float()
        class_probs  = torch.softmax(class_logits, dim=0).cpu().numpy()

        # Predicción: argmax sobre los logits restringidos a las 3 clases
        pred = CLASES[int(np.argmax(class_probs))]

        ids_out.append(row["id"])
        y_true_out.append(row["final_label"])
        preds_out.append(pred)
        probs_list.append(class_probs)

    del model
    torch.mps.empty_cache()

    probs_arr = np.array(probs_list)
    resultados = pd.DataFrame({
        "id":        ids_out,
        "true":      y_true_out,
        "pred":      preds_out,
        "p_izq":     probs_arr[:, 0],
        "p_cen":     probs_arr[:, 1],
        "p_der":     probs_arr[:, 2],
        "confianza": probs_arr.max(axis=1),
    })
    csv_path = OUT_ZS_DIR / f"zeroshot_{nombre_corto}_predictions.csv"
    resultados.to_csv(csv_path, index=False)
    print(f"  Predicciones → {csv_path}")
    print(f"  Fallbacks: 0/{len(df_muestra)} (argmax sobre logits, sin fallbacks)")

    metricas = calcular_metricas_zs(nombre_corto, y_true_out, preds_out, probs_arr)
    metricas["fallbacks_n"]   = 0
    metricas["fallbacks_pct"] = 0.0
    return metricas

In [25]:
# Ejecutar los 3 modelos Qwen3 de menor a mayor
todas_metricas_zs = []

# Baseline aleatorio (referencia)
todas_metricas_zs.append({
    "modelo": "Baseline random", "n_eval": 150,
    "accuracy": 0.33, "f1_macro": 0.33, "f1_weighted": 0.33, "kappa": 0.00,
    "confianza_media": float("nan"),
    "precision_izq": 0.33, "recall_izq": 0.33, "f1_izq": 0.33,
    "precision_cen": 0.33, "recall_cen": 0.33, "f1_cen": 0.33,
    "precision_der": 0.33, "recall_der": 0.33, "f1_der": 0.33,
    "conf_izq": float("nan"), "conf_cen": float("nan"), "conf_der": float("nan"),
    "fallbacks_n": 0, "fallbacks_pct": 0.0,
})

for modelo in ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]:
    todas_metricas_zs.append(evaluar_qwen3(modelo, df_muestra))

  Modelo: Qwen/Qwen3-0.6B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Predicciones → ../output/zeroshot_Qwen3-0.6B_predictions.csv
  Fallbacks: 0/150 (argmax sobre logits, sin fallbacks)

  Accuracy    : 0.3800
  F1-macro    : 0.3290
  F1-weighted : 0.3290
  Kappa       : 0.0700
  Confianza   : 0.8271
  IZQUIERDA: P=0.353  R=0.240  F1=0.286
  CENTRO: P=0.386  R=0.780  F1=0.517
  DERECHA: P=0.400  R=0.120  F1=0.185

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         12      30        8
CENTRO            10      39        1
DERECHA           12      32        6
  Modelo: Qwen/Qwen3-1.7B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Predicciones → ../output/zeroshot_Qwen3-1.7B_predictions.csv
  Fallbacks: 0/150 (argmax sobre logits, sin fallbacks)

  Accuracy    : 0.3133
  F1-macro    : 0.1591
  F1-weighted : 0.1591
  Kappa       : -0.0300
  Confianza   : nan
  IZQUIERDA: P=0.320  R=0.940  F1=0.477
  CENTRO: P=0.000  R=0.000  F1=0.000
  DERECHA: P=0.000  R=0.000  F1=0.000

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         47       1        2
CENTRO            50       0        0
DERECHA           50       0        0
  Modelo: Qwen/Qwen3-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  Predicciones → ../output/zeroshot_Qwen3-4B_predictions.csv
  Fallbacks: 0/150 (argmax sobre logits, sin fallbacks)

  Accuracy    : 0.3533
  F1-macro    : 0.3115
  F1-weighted : 0.3115
  Kappa       : 0.0300
  Confianza   : 0.7047
  IZQUIERDA: P=0.343  R=0.700  F1=0.461
  CENTRO: P=0.263  R=0.100  F1=0.145
  DERECHA: P=0.448  R=0.260  F1=0.329

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         35       8        7
CENTRO            36       5        9
DERECHA           31       6       13


In [9]:
# Recupera los resultados Qwen3 desde los CSV guardados — no re-ejecuta los modelos
todas_metricas_zs = []

todas_metricas_zs.append({
    "modelo": "Baseline random", "n_eval": 150,
    "accuracy": 0.33, "f1_macro": 0.33, "f1_weighted": 0.33, "kappa": 0.00,
    "confianza_media": float("nan"),
    "precision_izq": 0.33, "recall_izq": 0.33, "f1_izq": 0.33,
    "precision_cen": 0.33, "recall_cen": 0.33, "f1_cen": 0.33,
    "precision_der": 0.33, "recall_der": 0.33, "f1_der": 0.33,
    "conf_izq": float("nan"), "conf_cen": float("nan"), "conf_der": float("nan"),
    "fallbacks_n": 0, "fallbacks_pct": 0.0,
})

for nombre in ["Qwen3-0.6B", "Qwen3-1.7B", "Qwen3-4B"]:
    df_pred = pd.read_csv(OUT_ZS_DIR / f"zeroshot_{nombre}_predictions.csv")
    probs_arr = df_pred[["p_izq", "p_cen", "p_der"]].values
    m = calcular_metricas_zs(nombre, df_pred["true"].tolist(), df_pred["pred"].tolist(), probs_arr)
    m["fallbacks_n"]   = 0
    m["fallbacks_pct"] = 0.0
    todas_metricas_zs.append(m)

print(f"Recuperados {len(todas_metricas_zs)-1} modelos Qwen3 desde CSV")


  Accuracy    : 0.3800
  F1-macro    : 0.3290
  F1-weighted : 0.3290
  Kappa       : 0.0700
  Confianza   : 0.8271
  IZQUIERDA: P=0.353  R=0.240  F1=0.286
  CENTRO: P=0.386  R=0.780  F1=0.517
  DERECHA: P=0.400  R=0.120  F1=0.185

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         12      30        8
CENTRO            10      39        1
DERECHA           12      32        6

  Accuracy    : 0.3133
  F1-macro    : 0.1591
  F1-weighted : 0.1591
  Kappa       : -0.0300
  Confianza   : nan
  IZQUIERDA: P=0.320  R=0.940  F1=0.477
  CENTRO: P=0.000  R=0.000  F1=0.000
  DERECHA: P=0.000  R=0.000  F1=0.000

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         47       1        2
CENTRO            50       0        0
DERECHA           50       0        0

  Accuracy    : 0.3533
  F1-macro    : 0.3115
  F1-weighted : 0.3115
  Kappa       : 0.0300
  Confianza   : 0.7047
  IZQUIERDA: P=0.343  R=

**Zero-shot Qwen3 — análisis por modelo.**

- **Qwen3-0.6B** (F1-macro=0.329, Kappa=0.07): el mejor del grupo, pero estadísticamente equivalente al azar. Sesga hacia CENTRO: clasifica correctamente 39/50 artículos de esa clase (F1-CEN=0.517), pero falla sistemáticamente en IZQUIERDA (F1=0.286) y DERECHA (F1=0.185).

- **Qwen3-1.7B** (F1-macro=0.159, Kappa=−0.03): el peor resultado. Con `/no_think` colapsa prediciendo casi siempre IZQUIERDA — los 50 artículos de CENTRO y los 50 de DERECHA se clasifican como IZQUIERDA. El tamaño intermedio no aporta capacidad de razonamiento suficiente.

- **Qwen3-4B** (F1-macro=0.312, Kappa=0.03): distribución de predicciones más equilibrada, pero también cerca del azar. CENTRO apenas reconocido (F1-CEN=0.145). Necesita ejemplos en contexto para activar su potencial.

### 5.5 Evaluación RoBERTa-BNE (zero-shot classification pipeline)

In [10]:
subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece", "protobuf"], check=True)


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/Users/blancagallardo/PycharmProjects/TFM/.venv/bin/python', '-m', 'pip', 'install', 'sentencepiece', 'protobuf'], returncode=0)

In [11]:
def evaluar_roberta(df_muestra):
    nombre_corto = "xlm-roberta-large-xnli"
    print(f"\n{'='*64}")
    print(f"  Modelo: joeddav/xlm-roberta-large-xnli")
    print(f"{'='*64}")

    tokenizer_rob = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli", use_fast=False)
    clf = pipeline(
        "zero-shot-classification",
        model="joeddav/xlm-roberta-large-xnli",
        tokenizer=tokenizer_rob,
        device=ZS_DEVICE,
        torch_dtype=ZS_DTYPE,
    )

    ids_out, y_true_out, preds_out = [], [], []
    probs_list = []

    for _, row in df_muestra.iterrows():
        result = clf(
            row["texto"],
            candidate_labels=["izquierda", "centro", "derecha"],
            hypothesis_template="Este texto es de ideología {}.",
            truncation=True,
        )
        score_map = dict(zip(result["labels"], result["scores"]))
        probs = np.array([
            score_map.get("izquierda", 0.0),
            score_map.get("centro",    0.0),
            score_map.get("derecha",   0.0),
        ])
        pred = CLASES[int(np.argmax(probs))]

        ids_out.append(row["id"])
        y_true_out.append(row["final_label"])
        preds_out.append(pred)
        probs_list.append(probs)

    probs_arr = np.array(probs_list)
    resultados = pd.DataFrame({
        "id":        ids_out,
        "true":      y_true_out,
        "pred":      preds_out,
        "p_izq":     probs_arr[:, 0],
        "p_cen":     probs_arr[:, 1],
        "p_der":     probs_arr[:, 2],
        "confianza": probs_arr.max(axis=1),
    })
    csv_path = OUT_ZS_DIR / f"zeroshot_{nombre_corto}_predictions.csv"
    resultados.to_csv(csv_path, index=False)
    print(f"\n  Predicciones → {csv_path}")

    metricas = calcular_metricas_zs(nombre_corto, y_true_out, preds_out, probs_arr)
    metricas["fallbacks_n"]   = 0
    metricas["fallbacks_pct"] = 0.0
    return metricas

todas_metricas_zs.append(evaluar_roberta(df_muestra))


  Modelo: joeddav/xlm-roberta-large-xnli


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Predicciones → ../output/zeroshot_xlm-roberta-large-xnli_predictions.csv

  Accuracy    : 0.3400
  F1-macro    : 0.3249
  F1-weighted : 0.3249
  Kappa       : 0.0100
  Confianza   : 0.3786
  IZQUIERDA: P=0.387  R=0.240  F1=0.296
  CENTRO: P=0.354  R=0.560  F1=0.434
  DERECHA: P=0.275  R=0.220  F1=0.244

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         12      25       13
CENTRO             6      28       16
DERECHA           13      26       11


**RoBERTa NLI zero-shot** (F1-macro=0.325, Kappa=0.01): rendimiento prácticamente idéntico al azar. La hipótesis `"Este texto es de ideología {}"` no captura las sutilezas del contexto mediático español. Sesga hacia CENTRO (F1-CEN=0.434) y es el peor en DERECHA (F1-DER=0.244). Además, el pipeline NLI no admite ejemplos en el prompt, lo que limita su mejora en la fase few-shot posterior.

### 5.6 Tabla comparativa final y modelo ganador

In [12]:
df_zs_results = pd.DataFrame(todas_metricas_zs)
df_zs_results.to_csv(OUT_ZS_DIR / "zeroshot_results.csv", index=False)
print(f"Métricas globales → {OUT_ZS_DIR / 'zeroshot_results.csv'}")

cols_tabla = ["modelo", "accuracy", "f1_macro", "f1_izq", "f1_cen", "f1_der", "kappa"]
print("\n" + "="*80)
print("TABLA COMPARATIVA FINAL")
print("="*80)
print(df_zs_results[cols_tabla].to_string(index=False))

candidatos = df_zs_results[df_zs_results["modelo"] != "Baseline random"].copy()
max_f1     = candidatos["f1_macro"].max()
empate     = candidatos[candidatos["f1_macro"] == max_f1]

if len(empate) > 1:
    qwen_ties = empate[empate["modelo"].str.contains("Qwen", case=False)]
    ganador   = qwen_ties.iloc[0]["modelo"] if len(qwen_ties) > 0 else empate.iloc[0]["modelo"]
else:
    ganador = empate.iloc[0]["modelo"]

print(f"\n>>> MODELO GANADOR (F1-macro): {ganador}  →  {max_f1:.4f}")

Métricas globales → ../output/zeroshot_results.csv

TABLA COMPARATIVA FINAL
                modelo  accuracy  f1_macro  f1_izq  f1_cen  f1_der  kappa
       Baseline random    0.3300    0.3300  0.3300  0.3300  0.3300   0.00
            Qwen3-0.6B    0.3800    0.3290  0.2857  0.5166  0.1846   0.07
            Qwen3-1.7B    0.3133    0.1591  0.4772  0.0000  0.0000  -0.03
              Qwen3-4B    0.3533    0.3115  0.4605  0.1449  0.3291   0.03
xlm-roberta-large-xnli    0.3400    0.3249  0.2963  0.4341  0.2444   0.01

>>> MODELO GANADOR (F1-macro): Qwen3-0.6B  →  0.3290


**Conclusión zero-shot.** Ningún modelo supera el baseline aleatorio (F1-macro ≈ 0.33): Qwen3-0.6B es el mejor con F1-macro=0.329 y Kappa=0.07, apenas sobre el azar. Qwen3-1.7B muestra sesgo sistemático hacia IZQUIERDA (F1-CENTRO=F1-DERECHA=0.00), colapsando en una respuesta degenerada. Estos resultados confirman que la clasificación ideológica de prensa española requiere supervisión o al menos ejemplos en contexto: el conocimiento preentrenado no incluye el mapa ideológico de los medios españoles. Por ello se evalúa una configuración few-shot antes de decidir el modelo base para fine-tuning con LoRA.

## Sección 5.7 — Evaluación Few-Shot (2 ejemplos por clase)

Se seleccionan los **2 artículos más representativos de cada clase** del conjunto de train
(los más cercanos al centroide TF-IDF de su clase) y se incluyen como ejemplos en el prompt.

Se reutilizan: `df_muestra`, `calcular_metricas_zs`, `OUT_ZS_DIR`, `CLASES` y la misma muestra de 150 artículos.

In [18]:
df_train_fs = pd.read_parquet(SPLITS_DIR / 'df_train.parquet')

tfidf_fs = TfidfVectorizer(
    max_features=8000, ngram_range=(1, 2),
    stop_words=STOPWORDS_ES, min_df=2,
    strip_accents='unicode', lowercase=True,
)
X_train_fs = tfidf_fs.fit_transform(df_train_fs['texto'])

# 2 más cercanos al centroide TF-IDF por clase
ejemplos_fs = {}
for cls in CLASES:
    mask    = df_train_fs['final_label'] == cls
    idx_cls = np.where(mask)[0]
    X_cls   = X_train_fs[idx_cls]
    centroide = np.asarray(X_cls.mean(axis=0))
    sims    = cosine_similarity(X_cls, centroide).flatten()
    top2    = idx_cls[np.argsort(sims)[-2:][::-1]]
    ejemplos_fs[cls] = df_train_fs.iloc[top2]['texto'].tolist()
    print(f'{cls}:')
    for t in ejemplos_fs[cls]:
        print(f'  {t[:100]}...')


IZQUIERDA:
  Francina Armengol: "Dividir la izquierda es un disparate y no es lo que se merece la ciudadanía". La...
  Yolanda Díaz: "No puedo comprender por qué el PSOE quiere castigar a las mujeres trabajadoras". "El ...
CENTRO:
  Toni Roldán: "Los países con una jornada laboral corta se la pueden permitir porque tienen economías...
  Las compras de fondos soberanos en España marcan un récord en 2022: casi 3.000 millones en 11 operac...
DERECHA:
  Montero, Santos Cerdán, Torres El cerco judicial al Gobierno continúa tras Bolaños. El Supremo ya ha...
  Vicente Gil: «Si Óscar Puente especula con que Milei se droga, ¿se puede hacer lo mismo con Puente?....


**Selección de ejemplos few-shot.** Los 2 artículos por clase se eligen como los **más cercanos al centroide TF-IDF** de su clase en el corpus de entrenamiento. Esta estrategia selecciona ejemplos prototípicos — léxico ideológico concentrado y representativo — en lugar de artículos atípicos que podrían introducir ruido en el prompt. Cada clase aporta exactamente 2 ejemplos, manteniendo el balanceo en el contexto del modelo.

In [19]:
def _trunc(texto, max_chars=120):
    return texto[:max_chars] + '...' if len(texto) > max_chars else texto

def build_fewshot_prompt(texto):
    return (
        '/no_think\n'
        'Clasifica la ideología política de este artículo español.\n'
        'Responde con exactamente una palabra: IZQUIERDA, CENTRO o DERECHA.\n\n'
        'Ejemplos:\n\n'
        f'Artículo: {_trunc(ejemplos_fs["IZQUIERDA"][0])}\nRespuesta: izquierda\n\n'
        f'Artículo: {_trunc(ejemplos_fs["IZQUIERDA"][1])}\nRespuesta: izquierda\n\n'
        f'Artículo: {_trunc(ejemplos_fs["CENTRO"][0])}\nRespuesta: centro\n\n'
        f'Artículo: {_trunc(ejemplos_fs["CENTRO"][1])}\nRespuesta: centro\n\n'
        f'Artículo: {_trunc(ejemplos_fs["DERECHA"][0])}\nRespuesta: derecha\n\n'
        f'Artículo: {_trunc(ejemplos_fs["DERECHA"][1])}\nRespuesta: derecha\n\n'
        f'Artículo: {_trunc(texto, 500)}\n\n'
        'Respuesta (una palabra):'
    )

def evaluar_qwen3_fewshot(nombre_modelo, df_muestra):
    nombre_corto = nombre_modelo.split('/')[-1] + '_fewshot'
    print(f'\n{"="*64}')
    print(f'  Modelo (few-shot): {nombre_modelo}')
    print(f'{"="*64}')

    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    model = AutoModelForCausalLM.from_pretrained(
        nombre_modelo, torch_dtype=ZS_DTYPE,
    ).to(ZS_DEVICE)
    model.eval()

    tok_ids = [tokenizer.encode(w, add_special_tokens=False)[0]
               for w in ['izquierda', 'centro', 'derecha']]

    ids_out, y_true_out, preds_out, probs_list = [], [], [], []

    for _, row in df_muestra.iterrows():
        prompt = build_fewshot_prompt(row['texto'])
        inputs = tokenizer(
            prompt, return_tensors='pt',
            max_length=1024, truncation=True,
        ).to(ZS_DEVICE)

        with torch.no_grad():
            gen_out = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                temperature=None, top_p=None,
                output_scores=True, return_dict_in_generate=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        class_logits = gen_out.scores[0][0][tok_ids].float()
        class_probs  = torch.softmax(class_logits, dim=0).cpu().numpy()
        pred = CLASES[int(np.argmax(class_probs))]

        ids_out.append(row['id'])
        y_true_out.append(row['final_label'])
        preds_out.append(pred)
        probs_list.append(class_probs)

    del model
    torch.mps.empty_cache()

    probs_arr = np.array(probs_list)
    resultados = pd.DataFrame({
        'id': ids_out, 'true': y_true_out, 'pred': preds_out,
        'p_izq': probs_arr[:, 0], 'p_cen': probs_arr[:, 1],
        'p_der': probs_arr[:, 2], 'confianza': probs_arr.max(axis=1),
    })
    csv_path = OUT_ZS_DIR / f'fewshot_{nombre_corto}_predictions.csv'
    resultados.to_csv(csv_path, index=False)
    print(f'  Predicciones → {csv_path}')

    metricas = calcular_metricas_zs(nombre_corto, y_true_out, preds_out, probs_arr)
    metricas['fallbacks_n']   = 0
    metricas['fallbacks_pct'] = 0.0
    return metricas

In [20]:
todas_metricas_fs = []

for modelo in ['Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B', 'Qwen/Qwen3-4B']:
    todas_metricas_fs.append(evaluar_qwen3_fewshot(modelo, df_muestra))



  Modelo (few-shot): Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Predicciones → ../output/fewshot_Qwen3-0.6B_fewshot_predictions.csv

  Accuracy    : 0.3333
  F1-macro    : 0.1667
  F1-weighted : 0.1667
  Kappa       : 0.0000
  Confianza   : 0.9077
  IZQUIERDA: P=0.000  R=0.000  F1=0.000
  CENTRO: P=0.000  R=0.000  F1=0.000
  DERECHA: P=0.333  R=1.000  F1=0.500

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA          0       0       50
CENTRO             0       0       50
DERECHA            0       0       50

  Modelo (few-shot): Qwen/Qwen3-1.7B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Predicciones → ../output/fewshot_Qwen3-1.7B_fewshot_predictions.csv

  Accuracy    : 0.3600
  F1-macro    : 0.2824
  F1-weighted : 0.2824
  Kappa       : 0.0400
  Confianza   : 0.9588
  IZQUIERDA: P=0.400  R=0.080  F1=0.133
  CENTRO: P=0.875  R=0.140  F1=0.241
  DERECHA: P=0.326  R=0.860  F1=0.473

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA          4       0       46
CENTRO             0       7       43
DERECHA            6       1       43

  Modelo (few-shot): Qwen/Qwen3-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  Predicciones → ../output/fewshot_Qwen3-4B_fewshot_predictions.csv

  Accuracy    : 0.6200
  F1-macro    : 0.6254
  F1-weighted : 0.6254
  Kappa       : 0.4300
  Confianza   : 0.9045
  IZQUIERDA: P=0.688  R=0.440  F1=0.537
  CENTRO: P=0.939  R=0.620  F1=0.747
  DERECHA: P=0.471  R=0.800  F1=0.593

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         22       1       27
CENTRO             1      31       18
DERECHA            9       1       40


**Few-shot Qwen3: análisis por modelo**
 - **Qwen3-0.6B** (F1-macro=0.167, Δ=−0.162): **empeora drásticamente** respecto al zero-shot. Con los 6 ejemplos en el prompt colapsa prediciendo siempre DERECHA (150/150). El modelo pequeño no tiene capacidad para mantener coherencia en un prompt extendido.

 - **Qwen3-1.7B** (F1-macro=0.282, Δ=+0.123): leve mejora. Ya reconoce algo de CENTRO e IZQUIERDA, pero sigue sesgando hacia DERECHA (86/150 predicciones). Insuficiente para superar el azar.

- **Qwen3-4B** (F1-macro=0.625, Kappa=0.43, Δ=+0.314): **salto cualitativo**. Es el único modelo que supera claramente el azar. F1-CEN=0.747, F1-DER=0.593, F1-IZQ=0.537. Los 6 ejemplos contextuales son suficientes para que el modelo de 4B entienda la tarea ideológica.

### Few-shot RoBERTa

El pipeline NLI no soporta ejemplos en el prompt directamente.
Se evalúa con zero-shot estándar para mantener la comparativa completa.

In [21]:
def evaluar_roberta_fewshot(df_muestra):
    nombre_corto = 'xlm-roberta-xnli_fewshot'
    print(f'\n{"="*64}')
    print(f'  Modelo: joeddav/xlm-roberta-large-xnli  (NLI pipeline, sin few-shot)')
    print(f'{"="*64}')

    from transformers import AutoTokenizer as AT
    tokenizer_rob = AT.from_pretrained(
        'joeddav/xlm-roberta-large-xnli', use_fast=False
    )
    clf = pipeline(
        'zero-shot-classification',
        model='joeddav/xlm-roberta-large-xnli',
        tokenizer=tokenizer_rob,
        device=ZS_DEVICE, torch_dtype=ZS_DTYPE,
    )

    ids_out, y_true_out, preds_out, probs_list = [], [], [], []
    for _, row in df_muestra.iterrows():
        result = clf(
            row['texto'],
            candidate_labels=['izquierda', 'centro', 'derecha'],
            hypothesis_template='Este texto es de ideología {}.',
            truncation=True,
        )
        score_map = dict(zip(result['labels'], result['scores']))
        probs = np.array([score_map.get('izquierda', 0.0),
                          score_map.get('centro',    0.0),
                          score_map.get('derecha',   0.0)])
        preds_out.append(CLASES[int(np.argmax(probs))])
        ids_out.append(row['id'])
        y_true_out.append(row['final_label'])
        probs_list.append(probs)

    probs_arr = np.array(probs_list)
    pd.DataFrame({
        'id': ids_out, 'true': y_true_out, 'pred': preds_out,
        'p_izq': probs_arr[:, 0], 'p_cen': probs_arr[:, 1],
        'p_der': probs_arr[:, 2], 'confianza': probs_arr.max(axis=1),
    }).to_csv(OUT_ZS_DIR / f'fewshot_{nombre_corto}_predictions.csv', index=False)

    metricas = calcular_metricas_zs(nombre_corto, y_true_out, preds_out, probs_arr)
    metricas['fallbacks_n']   = 0
    metricas['fallbacks_pct'] = 0.0
    return metricas

todas_metricas_fs.append(evaluar_roberta_fewshot(df_muestra))



  Modelo: joeddav/xlm-roberta-large-xnli  (NLI pipeline, sin few-shot)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Accuracy    : 0.3400
  F1-macro    : 0.3249
  F1-weighted : 0.3249
  Kappa       : 0.0100
  Confianza   : 0.3786
  IZQUIERDA: P=0.387  R=0.240  F1=0.296
  CENTRO: P=0.354  R=0.560  F1=0.434
  DERECHA: P=0.275  R=0.220  F1=0.244

  Matriz de confusión (filas=real, cols=pred):
           IZQUIERDA  CENTRO  DERECHA
IZQUIERDA         12      25       13
CENTRO             6      28       16
DERECHA           13      26       11


In [22]:
# Guardar métricas few-shot
df_fs_results = pd.DataFrame(todas_metricas_fs)
df_fs_results.to_csv(OUT_ZS_DIR / 'fewshot_results.csv', index=False)

# Tabla comparativa few-shot vs zero-shot
cols = ['modelo', 'accuracy', 'f1_macro', 'f1_izq', 'f1_cen', 'f1_der', 'kappa']
print('='*80)
print('FEW-SHOT — TABLA COMPARATIVA')
print('='*80)
print(df_fs_results[cols].to_string(index=False))

# Zero-shot vs Few-shot por modelo
df_zs = pd.read_csv(OUT_ZS_DIR / 'zeroshot_results.csv')
print('\n' + '='*80)
print('ZERO-SHOT vs FEW-SHOT — F1-macro')
print('='*80)
for fs_row in df_fs_results.itertuples():
    base = fs_row.modelo.replace('_fewshot', '')
    zs_match = df_zs[df_zs['modelo'] == base]
    zs_f1 = zs_match['f1_macro'].values[0] if len(zs_match) > 0 else float('nan')
    delta = fs_row.f1_macro - zs_f1
    signo = '+' if delta >= 0 else ''
    print(f'  {base:<30} ZS={zs_f1:.4f}  FS={fs_row.f1_macro:.4f}  Δ={signo}{delta:.4f}')

# Ganador few-shot por F1-macro
ganador_fs = df_fs_results.loc[df_fs_results['f1_macro'].idxmax(), 'modelo']
print(f'\n>>> GANADOR FEW-SHOT: {ganador_fs}  →  {df_fs_results["f1_macro"].max():.4f}')


FEW-SHOT — TABLA COMPARATIVA
                  modelo  accuracy  f1_macro  f1_izq  f1_cen  f1_der  kappa
      Qwen3-0.6B_fewshot    0.3333    0.1667  0.0000  0.0000  0.5000   0.00
      Qwen3-1.7B_fewshot    0.3600    0.2824  0.1333  0.2414  0.4725   0.04
        Qwen3-4B_fewshot    0.6200    0.6254  0.5366  0.7470  0.5926   0.43
xlm-roberta-xnli_fewshot    0.3400    0.3249  0.2963  0.4341  0.2444   0.01

ZERO-SHOT vs FEW-SHOT — F1-macro
  Qwen3-0.6B                     ZS=0.3290  FS=0.1667  Δ=-0.1623
  Qwen3-1.7B                     ZS=0.1591  FS=0.2824  Δ=+0.1233
  Qwen3-4B                       ZS=0.3115  FS=0.6254  Δ=+0.3139
  xlm-roberta-xnli               ZS=nan  FS=0.3249  Δ=nan

>>> GANADOR FEW-SHOT: Qwen3-4B_fewshot  →  0.6254


**Comparativa ZS vs FS**
| Modelo | ZS F1-macro | FS F1-macro | Δ |
|---|:---:|:---:|:---:|
| Qwen3-0.6B | 0.329 | 0.167 | −0.162 |
| Qwen3-1.7B | 0.159 | 0.282 | +0.123 |
| **Qwen3-4B** | 0.312 | **0.625** | **+0.314** |
| RoBERTa NLI | 0.325 | 0.325 | 0.000 |

El few-shot solo beneficia a modelos con capacidad suficiente: Qwen3-4B obtiene la mayor ganancia (+0.31) y es el **único modelo que supera el azar** con un Kappa=0.43 (acuerdo moderado). Qwen3-0.6B empeora al no poder mantener coherencia con un prompt extendido. RoBERTa no cambia al no soportar ejemplos en el pipeline NLI.

**Modelo seleccionado para fine-tuning con LoRA: Qwen3-4B.** Aun así, su F1-macro few-shot (0.625) dista ~14 puntos del mejor modelo supervisado (TF-IDF + LR, 0.765), lo que refuerza la necesidad del fine-tuning.

---
# Conclusion

Se evaluó la clasificación ideológica de artículos de prensa española mediante modelos de lenguaje en configuraciones zero-shot y few-shot, sin realizar fine-tuning. Los resultados muestran que los enfoques zero-shot obtienen un rendimiento cercano al azar (F1-macro ≈ 0.33), lo que indica que el conocimiento preentrenado de los modelos no es suficiente para capturar las particularidades ideológicas del contexto mediático español.

La incorporación de ejemplos en contexto (few-shot) solo resultó efectiva en modelos de mayor tamaño. En particular, Qwen3-4B alcanzó un F1-macro de 0.625 y un Kappa de 0.43, superando claramente al resto de configuraciones. Por el contrario, los modelos más pequeños no lograron beneficiarse del aprendizaje en contexto e incluso mostraron un deterioro de rendimiento.

Aunque Qwen3-4B constituye la alternativa más prometedora dentro de los modelos evaluados, su desempeño sigue situándose por debajo del mejor enfoque supervisado obtenido previamente (TF-IDF + Regresión Logística, F1-macro = 0.765). Estos resultados sugieren que, si bien el aprendizaje en contexto permite capturar parte de la señal ideológica, continúa siendo insuficiente para igualar a los modelos entrenados específicamente para la tarea.

En consecuencia, se selecciona Qwen3-4B como modelo base para la fase de fine-tuning mediante LoRA, con el objetivo de mejorar su capacidad de generalización y alcanzar un rendimiento comparable o superior al baseline supervisado.